### This notebook compute "VSS6. Volume of annual water demand for population use" indicator for the 27 basins of IKI Project

Spanish: Volumen de la demanda anual del agua para uso poblacional

**Created:** 12/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/22/2025

**Status:** Complete (for baseline scenario)

**QA Status:** reviewed by Scott Sheeder  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad
 
**Inputs:**  waterALLOC output database

**Outputs:** 
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** Here we directly query the waterALLOC output SQL database and import the results to the indicators database. We have included some code to check the min and max values of the results and the min and max values listed in the indicators database to ensure they align. Code is provided to update the min and max values in the indicators DB as needed. 

In [1]:
#import numpy as np
import pandas as pd
import sqlite3
#import geopandas as gpd
from scipy.spatial import cKDTree
#import os
from tqdm import tqdm

In [2]:
#SETTINGS
#user = 'sbakar'
user = 'ssheeder'
ScnID = 1 #Scenario ID -- 1: Baseline, 1981-2005, 2: Future, 2030, 3: Future: 2050
IndID = 406 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
#db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
#wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in scenarios_df["Scenario"]:
    print(f" - {s}")

Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020
 - Linea_Base_2020_Embalses


In [4]:
# select scenario to use
scenario_name = "Linea_Base_2020" # change name based on available scenarios above

In [5]:
query_demanda = """
SELECT 
    a.comid AS COMID,
    a.[Demanda] AS Demanda,
    a.[Suministro] AS Suministro
FROM "WAMSS_Demanda anual promedio por tipo de demanda por COMID" AS a
JOIN WAMMS_RunsInfo AS b 
    ON a.RunID = b.RunID
JOIN Scenarios AS c 
    ON c.ScnID = b.ScnID
WHERE c.Scenario = ?
  AND a.Sector = ?
"""

sector_name = "Poblacional"

demanda_df = pd.read_sql_query(
    query_demanda,
    conn_wa,
    params=(scenario_name, sector_name)
)

# Close connection
conn_wa.close()

In [6]:
# get min and max from the demanda column
min_demanda = demanda_df['Demanda'].min()
max_demanda = demanda_df['Demanda'].max()
print(f"Min Demanda: {min_demanda}")
print(f"Max Demanda: {max_demanda}")

Min Demanda: 0.9499999999999998
Max Demanda: 43523.30000000003


In [7]:
# check in the indicators database to make sure that the min and max values align with what is in the Indicators Table
conn_ind = sqlite3.connect(db_path)

# Query Min and Max for selected Indicator ID
query_minmax = """
SELECT 
    IndID,
    Min,
    Max
FROM Indicators
WHERE IndID = ?
"""

minmax_df = pd.read_sql_query(
    query_minmax,
    conn_ind,
    params=(IndID,)
)

print(f"Indicator ID {IndID} Min from DB: {minmax_df['Min'].values[0]}")
print(f"Indicator ID {IndID} Max from DB: {minmax_df['Max'].values[0]}")

# Close connection
conn_ind.close()

Indicator ID 406 Min from DB: 0
Indicator ID 406 Max from DB: 50000


In [8]:
# optional - Update the min and max values using the min and max demanda values calculated above
conn_ind = sqlite3.connect(db_path)

new_min = 0 
new_max = 50000
conn_ind = sqlite3.connect(db_path)
update_query = """
UPDATE Indicators
SET Min = ?, 
    Max = ?
WHERE IndID = ?
"""

cursor = conn_ind.cursor()
cursor.execute(
    update_query,
    (new_min, new_max, IndID)
)

# check uf update was successful
check_query = """
SELECT IndID, Min, Max
FROM Indicators
WHERE IndID = ?
"""

after_df = pd.read_sql_query(
    check_query,
    conn_ind,
    params=(IndID,)
)

print("After update:")
print(after_df)


conn_ind.commit()

After update:
   IndID  Min    Max
0    406    0  50000


In [9]:
#Update relevant dataframe and value column from the calculations above for the specific indicator
insert_data= demanda_df
value_column= 'Demanda'

In [10]:
# Connect to your SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Fixed IndID and ScnID (update above)

#For now I am commenting out the looping of scenarios as I imagine we might run just baseline scenarios for now.
# Define the mapping between scenario and DataFrame column
# scenario_columns = {
#     1: 'ACTUAL ANUAL',
#     2: '2030 ANUAL',
#     3: '2050 ANUAL'
# }

# Prepare list of rows to insert
rows_to_insert = []

#for scn_id, column_name in scenario_columns.items():
for _, row in insert_data.iterrows():
    comid = row['COMID']

    # Get values for each indicator
    val = row[value_column]

    # Append both indicators, one row each
    rows_to_insert.append((ScnID, IndID, comid, val))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [11]:
# Check for duplicates in the input dataframe before insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [17]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()